# Bend You Can Measure — reproducible code

Companion notebook for the Big Data Bowl 2027 writeup *Bend You Can Measure: A Combine Turn-Force Score That Predicts NFL Pass-Rush Pressure*.

Every number and figure in the writeup is produced here. The notebook writes small modules to disk and runs them in order:

1. `data.py` / `features.py` — loading and the per-frame centripetal-acceleration (turn force) feature
2. `validate_dl.py` — robustness checks and in-game turn force on every DL pass rush
3. `final_metric.py` — the Combine Bend Score (CBS) and effect sizes in pressures per 100 rushes
4. `nested_test.py` — nested, leave-one-draft-class-out validation with permutations (DL and OL)
5. `composite_test.py` — includes the WR "brake-and-burst" idea that did **not** survive
6. `figures.py` — the six figures

In [ ]:
import importlib.util, subprocess, sys
for pkg in ['polars', 'statsmodels', 'scikit-learn']:
    mod = 'sklearn' if pkg == 'scikit-learn' else pkg
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

## Modules

In [ ]:
%%writefile data.py
"""Paths and loaders for the competition data."""
import os
from pathlib import Path

import polars as pl

KAGGLE_INPUT = Path("/kaggle/input")
if KAGGLE_INPUT.exists():  # running inside a Kaggle notebook: locate the competition files wherever they are mounted
    ROOT = Path("/kaggle/working")
    hits = sorted(KAGGLE_INPUT.rglob("combine_results.csv"))
    if not hits:
        raise FileNotFoundError("Competition data not found under /kaggle/input - add the NFL Big Data Bowl 2027 input.")
    RAW = hits[0].parent
else:
    ROOT = Path(__file__).resolve().parents[1]
    RAW = ROOT / "data" / "cache" / "competitions" / "nfl-big-data-bowl-2027" / "nfl-big-data-bowl-2027"
if os.environ.get("BDB_RAW"):  # local test run of the notebook
    RAW, ROOT = Path(os.environ["BDB_RAW"]), Path.cwd()
PROCESSED = ROOT / "data" / "processed"
NA = ["NA"]


def read(name: str, **kw) -> pl.DataFrame:
    return pl.read_csv(RAW / f"{name}.csv", null_values=NA, infer_schema_length=100_000, **kw)


def players() -> pl.DataFrame:
    p = read("players").with_columns(
        pl.col("draft_overall_pick").cast(pl.Int64, strict=False),
        pl.col("draft_round").cast(pl.Int64, strict=False),
    )
    return p.with_columns(undrafted=pl.col("draft_overall_pick").is_null())


def combine_results() -> pl.DataFrame:
    num = ["ten_yd_split", "forty", "vertical", "broad_jump", "three_cone",
           "short_shuttle", "bench_reps", "ngs_college_production_score", "ngs_final_score"]
    return read("combine_results").with_columns(pl.col(num).cast(pl.Float64, strict=False))


def combine_tracking(drill_type: str | None = None) -> pl.DataFrame:
    df = read("combine_tracking").filter(pl.col("entity_type") == "PLAYER")
    if drill_type:
        df = df.filter(pl.col("drill_type") == drill_type)
    return df.with_columns(pl.col("time").str.to_datetime()).sort("nfl_id", "drill_name", "attempt", "time")


def reg_season_plays() -> pl.DataFrame:
    games = read("games").select("game_id", "season", "season_type", "week")
    return read("player_play").join(games, on="game_id").filter(pl.col("season_type") == "REG")


In [ ]:
%%writefile features.py
"""Per-attempt movement traits from drill tracking: turning force (bend) and brake/re-acceleration at the cut."""
import numpy as np
import polars as pl


def _angdiff(d):
    """Frame-to-frame angle difference in degrees, wrapped to [-180, 180)."""
    return (np.diff(d) + 180) % 360 - 180


def turn_features(ct: pl.DataFrame) -> pl.DataFrame:
    """Per attempt (nfl_id, drill_name, attempt):
    - acel_c_max: peak sustained centripetal acceleration (s * omega, 3-frame moving average), yd/s²
    - bend_speed: mean speed on frames with a strong turn (omega > 60 deg/s and s > 2)
    - pre_s, min_s, post_s: speed before the main cut, at the cut and 1 s after
    - decel: speed lost per second into the cut; reaccel: speed gained per second after it
    - retain: post_s / pre_s
    """
    rows = []
    for (pid, drill, att), g in ct.group_by("nfl_id", "drill_name", "attempt"):
        s, d = g["s"].to_numpy(), g["dir"].to_numpy()
        if len(s) < 15:
            continue
        x, y = g["x"].to_numpy(), g["y"].to_numpy()
        # heading from the x/y path (more stable than 'dir' at low speed)
        heading = np.degrees(np.arctan2(np.gradient(y), np.gradient(x)))
        omega = np.abs(_angdiff(heading)) / 0.1  # graus/s
        omega = np.convolve(omega, np.ones(3) / 3, mode="same")
        sm = s[1:]
        ac = sm * np.radians(omega)
        curving = (omega > 60) & (sm > 2)
        # main cut: largest cumulative heading change within a 0.5 s window
        win = 5
        cum = np.array([np.abs(_angdiff(heading[i:i + win + 1])).sum() for i in range(len(heading) - win)])
        i_cut = int(np.argmax(cum)) + win // 2
        i_pre0, i_post1 = max(0, i_cut - 10), min(len(s) - 1, i_cut + 10)
        pre_s = s[i_pre0:i_cut + 1].max()
        i_min = i_pre0 + int(np.argmin(s[i_pre0:i_post1 + 1]))
        min_s = s[i_min]
        i_peak_pre = i_pre0 + int(np.argmax(s[i_pre0:i_min + 1]))
        post_s = s[min(len(s) - 1, i_min + 10)]
        rows.append(dict(
            nfl_id=pid, drill_name=drill, attempt=att,
            acel_c_max=float(np.max(np.convolve(ac, np.ones(3) / 3, mode="valid"))) if len(ac) >= 3 else None,
            bend_speed=float(sm[curving].mean()) if curving.sum() >= 3 else None,
            cut_angle=float(cum.max()),
            pre_s=float(pre_s), min_s=float(min_s), post_s=float(post_s),
            decel=float((pre_s - min_s) / max(0.1, (i_min - i_peak_pre) * 0.1)),
            reaccel=float((post_s - min_s) / 1.0),
            retain=float(post_s / pre_s) if pre_s > 0 else None,
        ))
    return pl.DataFrame(rows)


def centripetal(df: pl.DataFrame, keys: list[str], smooth: int = 3) -> pl.DataFrame:
    """Per-frame centripetal acceleration (yd/s²) from x, y; frames must be time-ordered within keys."""
    out = []
    for k, g in df.group_by(keys):
        x, y, s = g["x"].to_numpy(), g["y"].to_numpy(), g["s"].to_numpy()
        if len(x) < 6:
            continue
        h = np.unwrap(np.arctan2(np.gradient(y), np.gradient(x)))
        omega = np.abs(np.gradient(h)) / 0.1
        ac = np.convolve(s * omega, np.ones(smooth) / smooth, mode="same")
        out.append(dict(zip(keys, k)) | dict(ac_max=float(ac.max()), ac_p90=float(np.percentile(ac, 90)),
                                             smax=float(s.max())))
    return pl.DataFrame(out)



In [ ]:
%%writefile dl_viability.py
"""Failed idea #1 (reported for transparency): does pure burst in Combine DL drills relate to NFL get-off and pressure?"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
import numpy as np
import polars as pl
from scipy import stats
import statsmodels.formula.api as smf
import data

LO, HI = 1.0, 4.0  # speed window (yd/s) used to time the burst
DRILLS = ["FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION", "PASS_RUSH_DRILL", "FOUR_BAG_AGILITY_DRILL",
          "RUN_AND_CLUB_DRILL", "BODY_CONTROL_DRILL", "RUN_THE_HOOP_DRILL", "FORTY_YARD_DASH"]


def burst_features(ct: pl.DataFrame) -> pl.DataFrame:
    """Per attempt: time from first crossing LO to HI yd/s, and peak acceleration in that window."""
    ct = ct.with_columns(t=(pl.col("time") - pl.col("time").min()).over("nfl_id", "drill_name", "attempt")
                         .dt.total_milliseconds() / 1000)
    rows = []
    for (pid, drill, att), g in ct.group_by("nfl_id", "drill_name", "attempt"):
        s, a, t = g["s"].to_numpy(), g["a"].to_numpy(), g["t"].to_numpy()
        i_lo = np.argmax(s >= LO) if (s >= LO).any() else None
        if i_lo is None:
            continue
        after = np.nonzero(s[i_lo:] >= HI)[0]
        if not len(after):
            continue
        i_hi = i_lo + after[0]
        # linear interpolation to remove 0.1 s quantization
        def cross(i, lvl):
            if i == 0 or s[i] == s[i - 1]:
                return t[i]
            return t[i - 1] + (lvl - s[i - 1]) / (s[i] - s[i - 1]) * (t[i] - t[i - 1])
        rows.append(dict(nfl_id=pid, drill_name=drill, attempt=att,
                         rise=cross(i_hi, HI) - cross(i_lo, LO) if i_lo > 0 else None,
                         peak_a=a[i_lo:i_hi + 1].max(), smax=s.max()))
    return pl.DataFrame(rows)


if __name__ == "__main__":
    cr = data.combine_results()
    dl_ids = cr.filter(pl.col("combine_position") == "DL").select("nfl_id")
    ct = pl.concat([data.combine_tracking("SKILL_DRILLS_DL"), data.combine_tracking("FORTY_YARD_DASH")]) \
        .join(dl_ids, on="nfl_id").filter(pl.col("drill_name").is_in(DRILLS))
    att = burst_features(ct)
    print("attempts with burst:", att.height, " rise null (clip started moving):", att["rise"].null_count())
    feat = (att.group_by("nfl_id", "drill_name")
            .agg(rise=pl.col("rise").min(), peak_a=pl.col("peak_a").max(), smax=pl.col("smax").max()))
    wide = feat.pivot(on="drill_name", index="nfl_id", values=["rise", "peak_a"])

    # In-game outcomes (regular season)
    pp = data.reg_season_plays().join(dl_ids, on="nfl_id")
    rush = pp.filter(pl.col("player_get_off").is_not_null())
    game = rush.group_by("nfl_id").agg(
        snaps=pl.len(), get_off=pl.col("player_get_off").mean(),
        pressures=pl.col("time_to_pressure").is_not_null().sum(),
        sacks=pl.col("sack").fill_null(0).sum())
    lg_rate = game["pressures"].sum() / game["snaps"].sum()
    K = 100  # pseudo-rushes for shrinkage (simple empirical Bayes)
    game = game.with_columns(press_rate=(pl.col("pressures") + K * lg_rate) / (pl.col("snaps") + K))
    print(f"league pressure rate per rush snap (DL rookies): {lg_rate:.3f}")

    pl_ = data.players().select("nfl_id", "draft_overall_pick", "undrafted", "nfl_position")
    df = (game.filter(pl.col("snaps") >= 50)
          .join(wide, on="nfl_id", how="left")
          .join(cr.select("nfl_id", "ten_yd_split", "forty", "combine_weight", "ngs_athleticism_score"), on="nfl_id")
          .join(pl_, on="nfl_id")
          .with_columns(pick=pl.col("draft_overall_pick").fill_null(260)))
    print("players in analysis:", df.height, " undrafted:", df["undrafted"].sum())

    outcomes = ["get_off", "press_rate"]
    cands = [c for c in df.columns if c.startswith(("rise_", "peak_a_"))] + ["ten_yd_split", "forty", "ngs_athleticism_score", "pick"]
    print(f"\n{'feature':70s} n   rho(get_off)  rho(press_rate)")
    for c in cands:
        sub = df.select(c, *outcomes).drop_nulls()
        if sub.height < 30:
            continue
        r = [stats.spearmanr(sub[c], sub[o])[0] for o in outcomes]
        print(f"{c:70s} {sub.height:3d}  {r[0]:+.2f}         {r[1]:+.2f}")

    # Incremental test: does the burst trait add anything beyond pick, weight and the 10-yd split?
    pdf = df.to_pandas()
    pdf["log_pick"] = np.log(pdf["pick"])
    best = "rise_FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION"
    for o in outcomes:
        for base in ["log_pick + combine_weight", "log_pick + combine_weight + ten_yd_split"]:
            d = pdf[[o, best, "log_pick", "combine_weight", "ten_yd_split"]].dropna()
            z = (d - d.mean()) / d.std()
            m0 = smf.ols(f"{o} ~ {base}", z).fit()
            m1 = smf.ols(f"{o} ~ {base} + {best}", z).fit()
            print(f"\n[{o}] base: {base}  n={len(d)}  R2 {m0.rsquared:.3f} -> {m1.rsquared:.3f}  "
                  f"beta={m1.params[best]:+.2f} p={m1.pvalues[best]:.3f}")


In [ ]:
%%writefile validate_dl.py
"""Turn-force (DL) validation:
1. robustness to drill choice (incl. only the drills that did NOT stand out in the screen)
2. not just speed (control for max speed; smoothing / percentile variants)
3. the same measure in games: Combine -> game transfer and link to pressure
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
import numpy as np
import polars as pl
import statsmodels.formula.api as smf
import data
from features import centripetal

PROC = data.PROCESSED
PROC.mkdir(parents=True, exist_ok=True)
CORE = ["FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION", "RUN_THE_HOOP_DRILL", "RUN_AND_CLUB_DRILL",
        "FOUR_BAG_AGILITY_DRILL", "PASS_RUSH_DRILL", "BODY_CONTROL_DRILL"]
SCREEN_TOP = CORE[:3]   # the 3 drills at the top of the initial screen
BLIND = CORE[3:]        # the 3 drills that did NOT stand out


def zcols(df, cols):
    return df.with_columns([((pl.col(c) - pl.col(c).mean()) / pl.col(c).std()).alias(c) for c in cols])


def fit(d, outcome, base, x, label):
    d = d.select(outcome, x, *base).drop_nulls().to_pandas()
    z = (d - d.mean()) / d.std()
    m = smf.ols(f"{outcome} ~ {' + '.join(base + [x])}", z).fit()
    print(f"  {label:58s} n={len(d):3d} beta={m.params[x]:+.2f} p={m.pvalues[x]:.4f}")
    return m


cr = data.combine_results()
dl_ids = cr.filter(pl.col("combine_position") == "DL").select("nfl_id")
plr = data.players().select("nfl_id", "draft_overall_pick", "draft_year", "display_name", "nfl_position")

# ---------- Combine ----------
ct = data.combine_tracking("SKILL_DRILLS_DL").join(dl_ids, on="nfl_id").filter(pl.col("drill_name").is_in(CORE))
variants = {}
for sm in (1, 3, 5):
    a = centripetal(ct, ["nfl_id", "drill_name", "attempt"], smooth=sm)
    variants[sm] = a.group_by("nfl_id", "drill_name").agg(pl.col("ac_max", "ac_p90", "smax").max())
att = variants[3]


def index(att, drills, col="ac_max"):
    w = att.filter(pl.col("drill_name").is_in(drills)).pivot(on="drill_name", index="nfl_id", values=col)
    w = zcols(w, [c for c in w.columns if c != "nfl_id"])
    return w.select("nfl_id", pl.mean_horizontal([c for c in w.columns if c != "nfl_id"]).alias("idx"))


speed = index(att, CORE, "smax").rename({"idx": "speed_idx"})

# ---------- games: pressure (same outcome as composite_test) ----------
reg = data.reg_season_plays().join(dl_ids, on="nfl_id")
rush = reg.filter(pl.col("player_get_off").is_not_null()).with_columns(
    edge=pl.col("lined_up_position").is_in(["EDGE", "OLB"]).cast(pl.Int8),
    pressed=pl.col("time_to_pressure").is_not_null().cast(pl.Int8))
rush = rush.with_columns(press_adj=pl.col("pressed") - pl.col("pressed").mean().over("lined_up_position"))
g = (rush.group_by("nfl_id").agg(snaps=pl.len(), edge_share=pl.col("edge").mean(), s=pl.col("press_adj").sum(),
                                 pressures=pl.col("pressed").sum())
     .with_columns(press_adj=pl.col("s") / (pl.col("snaps") + 100)).filter(pl.col("snaps") >= 50))
base_df = (g.join(cr.select("nfl_id", "combine_weight", "ten_yd_split"), on="nfl_id").join(plr, on="nfl_id")
           .with_columns(log_pick=pl.col("draft_overall_pick").fill_null(260).log()))
BASE = ["log_pick", "combine_weight", "edge_share"]

print("\n[1] Robustness to drill choice")
for lab, drills in [("all 6 drills", CORE), ("only the 3 top-of-screen drills", SCREEN_TOP),
                    ("only the 3 drills that did NOT stand out (blind)", BLIND)]:
    fit(base_df.join(index(att, drills), on="nfl_id", how="left"), "press_adj", BASE, "idx", lab)
for d in CORE:
    fit(base_df.join(index(att, [d]), on="nfl_id", how="left"), "press_adj", BASE, "idx", f"  single drill: {d[:40]}")

print("\n[2] Not just speed")
full = base_df.join(index(att, CORE), on="nfl_id", how="left").join(speed, on="nfl_id", how="left")
fit(full, "press_adj", BASE, "speed_idx", "max speed alone")
fit(full, "press_adj", BASE + ["speed_idx"], "idx", "turn force | controlling for max speed")
for sm in (1, 5):
    fit(base_df.join(index(variants[sm], CORE), on="nfl_id", how="left"), "press_adj", BASE, "idx",
        f"smoothing {sm} frames")
fit(base_df.join(index(att, CORE, "ac_p90"), on="nfl_id", how="left"), "press_adj", BASE, "idx", "90th percentile instead of max")
fit(full, "press_adj", BASE + ["ten_yd_split"], "idx", "turn force | + 10-yd split")

# ---------- [2b] the same checks for the final CBS (3 arc drills) ----------
ARC = SCREEN_TOP
print("\n[2b] Final Combine Bend Score (3 arc drills)")
speed_arc = index(att, ARC, "smax").rename({"idx": "speed_idx"})
cbs = base_df.join(index(att, ARC), on="nfl_id", how="left").join(speed_arc, on="nfl_id", how="left")
fit(cbs, "press_adj", BASE, "idx", "CBS")
fit(cbs, "press_adj", BASE + ["speed_idx"], "idx", "CBS | controlling for max speed in the same drills")
for sm in (1, 5):
    fit(base_df.join(index(variants[sm], ARC), on="nfl_id", how="left"), "press_adj", BASE, "idx",
        f"CBS with smoothing {sm} frames")
fit(base_df.join(index(att, ARC, "ac_p90"), on="nfl_id", how="left"), "press_adj", BASE, "idx",
    "CBS from 90th percentile instead of peak")
pdf = cbs.select("press_adj", "idx", *BASE, "draft_year").drop_nulls().to_pandas()
for yr in sorted(pdf["draft_year"].unique()):
    tr, te = pdf[pdf["draft_year"] != yr], pdf[pdf["draft_year"] == yr]
    m0 = smf.ols(f"press_adj ~ {' + '.join(BASE)}", tr).fit()
    m1 = smf.ols(f"press_adj ~ {' + '.join(BASE)} + idx", tr).fit()
    e0, e1 = ((te.press_adj - m0.predict(te)) ** 2).mean(), ((te.press_adj - m1.predict(te)) ** 2).mean()
    print(f"  CBS holdout {yr}: n={len(te):2d}  beta_train={m1.params['idx']:+.4f}  MSE reduction={1 - e1 / e0:+.1%}")

# ---------- [3] turn force INSIDE games ----------
print("\n[3] In-game turn force (0-3 s after the snap, pass-rush snaps)")
keys = rush.select("game_id", "play_id", "nfl_id").unique()
frames = []
for yr in (2023, 2024, 2025):
    lf = (pl.scan_csv(data.RAW / f"game_tracking_{yr}.csv", null_values=data.NA, infer_schema_length=10000)
          .select("game_id", "play_id", "nfl_id", "time", "x", "y", "s", "event")
          .join(keys.lazy(), on=["game_id", "play_id", "nfl_id"]))
    frames.append(lf.collect())
gt = pl.concat(frames).with_columns(pl.col("time").str.to_datetime()).sort("game_id", "play_id", "nfl_id", "time")
snap = gt.filter(pl.col("event") == "ball_snap").group_by("game_id", "play_id", "nfl_id").agg(t0=pl.col("time").min())
gt = gt.join(snap, on=["game_id", "play_id", "nfl_id"]).with_columns(
    dt=(pl.col("time") - pl.col("t0")).dt.total_milliseconds() / 1000).filter(pl.col("dt").is_between(0, 3.0))
print(f"  rush frames: {gt.height:,}  rushes: {gt.select('game_id', 'play_id', 'nfl_id').n_unique():,}")
play_ac = centripetal(gt, ["game_id", "play_id", "nfl_id"], smooth=3)
play_ac.write_parquet(PROC / "dl_rush_play_turnforce.parquet")
game_tf = play_ac.group_by("nfl_id").agg(game_turn=pl.col("ac_max").mean(), game_turn_n=pl.len())
# play level: turn force on a rush vs pressure on that rush
pj = play_ac.join(rush.select("game_id", "play_id", "nfl_id", "pressed", "lined_up_position"),
                  on=["game_id", "play_id", "nfl_id"])
pdf = pj.to_pandas()
m = smf.logit("pressed ~ ac_max + C(lined_up_position)", pdf).fit(disp=0)
print(f"  play level: logit pressure ~ ac_max  beta={m.params['ac_max']:+.3f} p={m.pvalues['ac_max']:.2e} (n={len(pdf):,})")
tr = base_df.join(index(att, CORE), on="nfl_id", how="left").join(game_tf, on="nfl_id", how="left")
s = tr.select("idx", "game_turn").drop_nulls()
print(f"  Combine -> game transfer: r={np.corrcoef(s['idx'], s['game_turn'])[0, 1]:+.2f} (n={s.height})")
fit(tr, "game_turn", BASE, "idx", "in-game turn force ~ Combine index (with controls)")
fit(tr, "press_adj", BASE, "game_turn", "pressure ~ in-game turn force (with controls)")

tr.join(index(att, CORE).rename({"idx": "turn_force"}), on="nfl_id", how="left") \
  .write_parquet(PROC / "dl_player_table.parquet")
print("\nTop 12 by Combine turn force (with pick and pressure):")
with pl.Config(tbl_rows=12, tbl_width_chars=200):
    print(tr.sort("idx", descending=True).select("display_name", "nfl_position", "draft_overall_pick",
                                                 "idx", "pressures", "snaps", "press_adj").head(12))


In [ ]:
%%writefile final_metric.py
"""Combine Bend Score (CBS): final metric, effect sizes in football units and the player table."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
import numpy as np
import polars as pl
import statsmodels.formula.api as smf
import data
from features import centripetal

ARC = ["FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION", "RUN_THE_HOOP_DRILL", "RUN_AND_CLUB_DRILL"]
PROC = data.PROCESSED
PROC.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(2027)

cr = data.combine_results()
dl_ids = cr.filter(pl.col("combine_position") == "DL").select("nfl_id")
plr = data.players()

# --- Combine metric (every DL with tracking, not only those who played) ---
ct = data.combine_tracking("SKILL_DRILLS_DL").join(dl_ids, on="nfl_id").filter(pl.col("drill_name").is_in(ARC))
a = centripetal(ct, ["nfl_id", "drill_name", "attempt"], smooth=3)
a.write_parquet(PROC / "dl_combine_attempts.parquet")
best = a.group_by("nfl_id", "drill_name").agg(pl.col("ac_max").max())
print("peak ac by drill (yd/s²):", best.group_by("drill_name").agg(
    pl.col("ac_max").mean().round(2).alias("mean"), pl.col("ac_max").std().round(2).alias("sd")).rows())
w = best.pivot(on="drill_name", index="nfl_id", values="ac_max")
w = w.with_columns([((pl.col(c) - pl.col(c).mean()) / pl.col(c).std()).alias(f"z_{c}") for c in ARC])
w = w.with_columns(n_arc=pl.sum_horizontal([pl.col(c).is_not_null().cast(pl.Int8) for c in ARC]),
                   cbs_z=pl.mean_horizontal([f"z_{c}" for c in ARC]))
# friendly scale: 50 = average, 10 = 1 SD
w = w.with_columns(cbs=50 + 10 * (pl.col("cbs_z") - pl.col("cbs_z").mean()) / pl.col("cbs_z").std())
print(f"DL with a CBS: {w.filter(pl.col('cbs').is_not_null()).height} of {dl_ids.height}")

# --- in-game outcome ---
reg = data.reg_season_plays().join(dl_ids, on="nfl_id")
rush = reg.filter(pl.col("player_get_off").is_not_null()).with_columns(
    edge=pl.col("lined_up_position").is_in(["EDGE", "OLB"]).cast(pl.Int8),
    pressed=pl.col("time_to_pressure").is_not_null().cast(pl.Int8),
    sacked=pl.col("sack").fill_null(0))
rate_by_role = rush.group_by("lined_up_position").agg(pl.col("pressed").mean().alias("role_rate"))
rush = rush.join(rate_by_role, on="lined_up_position")
K = 100
g = (rush.group_by("nfl_id").agg(snaps=pl.len(), pressures=pl.col("pressed").sum(), sacks=pl.col("sacked").sum(),
                                 edge_share=pl.col("edge").mean(), exp_press=pl.col("role_rate").sum(),
                                 seasons=pl.col("season").n_unique())
     .with_columns(press_rate=pl.col("pressures") / pl.col("snaps"),
                   # pressures over expected for the alignment, per 100 rushes, shrunk
                   poe100=100 * (pl.col("pressures") - pl.col("exp_press")) / (pl.col("snaps") + K)))

tab = (plr.join(cr.select("nfl_id", "combine_weight", "ten_yd_split", "forty"), on="nfl_id")
       .join(dl_ids, on="nfl_id").join(w.select("nfl_id", "cbs", "cbs_z", "n_arc", *ARC), on="nfl_id", how="left")
       .join(g, on="nfl_id", how="left")
       .with_columns(log_pick=pl.col("draft_overall_pick").fill_null(260).log()))
tab.write_parquet(PROC / "dl_cbs_table.parquet")

an = tab.filter(pl.col("snaps") >= 50, pl.col("cbs").is_not_null()).to_pandas()
print(f"\nAnalysis: n={len(an)} (undrafted: {an['undrafted'].sum()})")

# --- effect size in football units ---
m = smf.wls("poe100 ~ cbs + log_pick + combine_weight + edge_share", an, weights=an["snaps"]).fit()
b, lo, hi = m.params["cbs"], *m.conf_int().loc["cbs"]
print(f"+10 CBS points (1 SD) -> {10*b:+.2f} pressures over expected per 100 rushes  95% CI [{10*lo:+.2f}, {10*hi:+.2f}]  p={m.pvalues['cbs']:.4f}")
lg = rush["pressed"].mean()
print(f"baseline pressure rate: {100*lg:.1f} per 100 rushes -> relative effect {10*b/(100*lg):+.0%}")
m2 = smf.wls("poe100 ~ cbs + log_pick + combine_weight + edge_share + ten_yd_split", an.dropna(subset=['ten_yd_split']),
             weights=an.dropna(subset=['ten_yd_split'])["snaps"]).fit()
print(f"  + 10yd split: {10*m2.params['cbs']:+.2f}  p={m2.pvalues['cbs']:.4f}  (n={int(m2.nobs)})")
# comparison: what draft slot is worth in the same model
print(f"  comparison: pick 10 vs pick 100 -> {m.params['log_pick']*(np.log(10)-np.log(100)):+.2f} per 100 rushes")

# quartiles (pooled, alignment-adjusted)
an["q"] = np.digitize(an["cbs"], np.percentile(an["cbs"], [25, 50, 75])) + 1
qt = an.groupby("q").apply(lambda d: 100 * (d["pressures"].sum() - d["exp_press"].sum()) / d["snaps"].sum(),
                           include_groups=False)
print("pressures over expected per 100 rushes, by CBS quartile:", qt.round(2).to_dict())
boot = []
for _ in range(2000):
    s = an.sample(len(an), replace=True, random_state=int(rng.integers(1e9)))
    f = lambda d: 100 * (d["pressures"].sum() - d["exp_press"].sum()) / d["snaps"].sum()
    boot.append(f(s[s["q"] == 4]) - f(s[s["q"] == 1]))
print(f"Q4 - Q1: {qt[4]-qt[1]:+.2f}  95% bootstrap CI [{np.percentile(boot, 2.5):+.2f}, {np.percentile(boot, 97.5):+.2f}]")

# --- hidden talent: high CBS and late pick / undrafted ---
hid = (tab.filter(pl.col("cbs").is_not_null())
       .filter((pl.col("cbs") >= 60) & ((pl.col("draft_overall_pick") > 100) | pl.col("undrafted")))
       .sort("cbs", descending=True)
       .select("display_name", "nfl_position", "draft_year", "draft_overall_pick", "cbs", "snaps", "pressures", "poe100"))
print("\nCBS >= 60 with pick > 100 or undrafted:")
with pl.Config(tbl_rows=30, tbl_width_chars=200):
    print(hid)
print("\nTop 10 CBS (all):")
with pl.Config(tbl_rows=10, tbl_width_chars=200):
    print(tab.filter(pl.col("cbs").is_not_null()).sort("cbs", descending=True)
          .select("display_name", "nfl_position", "draft_overall_pick", "cbs", "snaps", "pressures", "poe100").head(10))


In [ ]:
%%writefile nested_test.py
"""Nested test: drills are selected using ONLY the training draft classes; the test class is never seen.
The same protocol is also run for offensive linemen (OL) as a comparison."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
import numpy as np
import polars as pl
import statsmodels.formula.api as smf
import data
from features import centripetal

rng = np.random.default_rng(7)


def drill_table(drill_type, ids, min_players=60):
    ct = data.combine_tracking(drill_type).join(ids, on="nfl_id")
    keep = ct.group_by("drill_name").agg(pl.col("nfl_id").n_unique().alias("n")).filter(pl.col("n") >= min_players)
    ct = ct.join(keep.select("drill_name"), on="drill_name")
    a = centripetal(ct, ["nfl_id", "drill_name", "attempt"], smooth=3)
    return a.group_by("nfl_id", "drill_name").agg(pl.col("ac_max").max()) \
            .pivot(on="drill_name", index="nfl_id", values="ac_max")


def nested(df, drills, outcome, base, label, n_perm=300):
    pdf = df.to_pandas()
    years = sorted(pdf["draft_year"].unique())

    def run(p):
        preds, picks = [], []
        for yr in years:
            tr, te = p[p["draft_year"] != yr].copy(), p[p["draft_year"] == yr].copy()
            mu, sd = tr[drills].mean(), tr[drills].std()
            trz, tez = (tr[drills] - mu) / sd, (te[drills] - mu) / sd
            chosen = []
            for d in drills:
                dd = tr[[outcome] + base].assign(x=trz[d]).dropna()
                m = smf.ols(f"{outcome} ~ {' + '.join(base)} + x", dd).fit()
                if m.params["x"] > 0 and m.pvalues["x"] < 0.10:
                    chosen.append(d)
            if not chosen:  # blind fallback rule: use every drill
                chosen = drills
            picks.append((yr, len(chosen)))
            tr["idx"], te["idx"] = trz[chosen].mean(axis=1), tez[chosen].mean(axis=1)
            tr, te = tr.dropna(subset=[outcome, "idx"] + base), te.dropna(subset=[outcome, "idx"] + base)
            m0 = smf.ols(f"{outcome} ~ {' + '.join(base)}", tr).fit()
            m1 = smf.ols(f"{outcome} ~ {' + '.join(base)} + idx", tr).fit()
            preds.append(np.c_[te[outcome], m0.predict(te), m1.predict(te)])
        P = np.vstack(preds)
        return 1 - ((P[:, 0] - P[:, 2]) ** 2).sum() / ((P[:, 0] - P[:, 1]) ** 2).sum(), picks

    gain, picks = run(pdf)
    null = []
    for _ in range(n_perm):
        q = pdf.copy()
        q[drills] = q[drills].values[rng.permutation(len(q))]
        null.append(run(q)[0])
    p = (np.array(null) >= gain).mean()
    print(f"{label:34s} n={len(pdf):3d} drills={len(drills):2d}  out-of-sample MSE reduction={gain:+.1%}  "
          f"p_perm={p:.3f}  chosen per fold={picks}")
    return dict(gain=float(gain), p_perm=float(p), null=[float(v) for v in null])


if __name__ == "__main__":
    cr = data.combine_results()
    plr = data.players().select("nfl_id", "draft_overall_pick", "draft_year")
    reg = data.reg_season_plays()
    wt = cr.select("nfl_id", "combine_weight")

    def finish(g, ids, drill_type):
        tab = drill_table(drill_type, ids)
        drills = [c for c in tab.columns if c != "nfl_id"]
        df = (g.join(tab, on="nfl_id", how="left").join(wt, on="nfl_id").join(plr, on="nfl_id")
              .with_columns(log_pick=pl.col("draft_overall_pick").fill_null(260).log()))
        return df, drills

    # DL: pressure
    ids = cr.filter(pl.col("combine_position") == "DL").select("nfl_id")
    r = reg.join(ids, on="nfl_id").filter(pl.col("player_get_off").is_not_null()).with_columns(
        edge=pl.col("lined_up_position").is_in(["EDGE", "OLB"]).cast(pl.Int8),
        pressed=pl.col("time_to_pressure").is_not_null().cast(pl.Int8))
    r = r.with_columns(y=pl.col("pressed") - pl.col("pressed").mean().over("lined_up_position"))
    g = (r.group_by("nfl_id").agg(n=pl.len(), edge_share=pl.col("edge").mean(), s=pl.col("y").sum())
         .with_columns(y=pl.col("s") / (pl.col("n") + 100)).filter(pl.col("n") >= 50))
    df, drills = finish(g, ids, "SKILL_DRILLS_DL")
    res = nested(df, drills, "y", ["log_pick", "combine_weight", "edge_share"], "DL turn force -> pressure", n_perm=1000)
    import json
    (data.PROCESSED / "nested_dl.json").write_text(json.dumps(res))

    # OL: pressure allowed (lower is better -> use the negative)
    ids = cr.filter(pl.col("combine_position") == "OL").select("nfl_id")
    o = reg.join(ids, on="nfl_id").filter(pl.col("pressure_allowed").is_not_null()).with_columns(
        pa=pl.col("pressure_allowed").cast(pl.Int8))
    o = o.with_columns(y=-(pl.col("pa") - pl.col("pa").mean().over("lined_up_position")))
    g = (o.group_by("nfl_id").agg(n=pl.len(), tackle_share=(pl.col("lined_up_position") == "T").mean(),
                                  s=pl.col("y").sum())
         .with_columns(y=pl.col("s") / (pl.col("n") + 200)).filter(pl.col("n") >= 50))
    df, drills = finish(g, ids, "SKILL_DRILLS_OL")
    nested(df, drills, "y", ["log_pick", "combine_weight", "tackle_share"], "OL turn force -> pressure avoided")



In [ ]:
%%writefile composite_test.py
"""Composite indices (DL: turn force; WR: brake-and-burst), each the mean z-score across several drills.
Validation: OLS with controls, 5,000 permutations, and leave-one-draft-class-out prediction.
The WR idea is reported because it FAILED once its single strongest route was removed."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
import numpy as np
import polars as pl
import statsmodels.formula.api as smf
import data
from features import turn_features

rng = np.random.default_rng(27)
cr = data.combine_results()
pl_ = data.players().select("nfl_id", "draft_overall_pick", "draft_year")
reg = data.reg_season_plays()


def zmean(df: pl.DataFrame, cols: list[str], signs: dict[str, int]) -> pl.Series:
    z = [((pl.col(c) - pl.col(c).mean()) / pl.col(c).std() * signs.get(c, 1)) for c in cols]
    return df.select(pl.mean_horizontal(z)).to_series()


def per_drill(ct, ids, traits):
    f = turn_features(ct.join(ids, on="nfl_id"))
    return f.group_by("nfl_id", "drill_name").agg([pl.col(t).median() for t in traits])


def evaluate(df: pl.DataFrame, outcome: str, base: list[str], idx: str, label: str):
    d = df.select(outcome, idx, *base, "draft_year").drop_nulls().to_pandas()
    cols = [outcome, idx] + base
    z = d.copy(); z[cols] = (d[cols] - d[cols].mean()) / d[cols].std()
    f = f"{outcome} ~ {' + '.join(base)} + {idx}"
    m = smf.ols(f, z).fit()
    b_obs = m.params[idx]
    # permute the index: null distribution of the coefficient
    null = []
    for _ in range(5000):
        zz = z.copy(); zz[idx] = rng.permutation(zz[idx].values)
        null.append(smf.ols(f, zz).fit().params[idx])
    p_perm = (np.abs(null) >= abs(b_obs)).mean()
    # leave one draft class out
    gains = []
    for yr in sorted(d["draft_year"].unique()):
        tr, te = z[d["draft_year"] != yr], z[d["draft_year"] == yr]
        m0 = smf.ols(f"{outcome} ~ {' + '.join(base)}", tr).fit()
        m1 = smf.ols(f, tr).fit()
        e0 = ((te[outcome] - m0.predict(te)) ** 2).mean()
        e1 = ((te[outcome] - m1.predict(te)) ** 2).mean()
        gains.append((yr, len(te), 1 - e1 / e0, m1.params[idx]))
    print(f"\n=== {label} | n={len(d)} | {f}")
    print(f"beta={b_obs:+.3f} (z)  p_ols={m.pvalues[idx]:.4f}  p_perm={p_perm:.4f}  "
          f"CI95=[{m.conf_int().loc[idx, 0]:+.2f}, {m.conf_int().loc[idx, 1]:+.2f}]")
    for yr, n, g, b in gains:
        print(f"  holdout {yr}: n={n:2d}  MSE reduction={g:+.1%}  beta_train={b:+.2f}")


# ---------------- DL: turn force ----------------
dl_ids = cr.filter(pl.col("combine_position") == "DL").select("nfl_id")
DL_DRILLS = ["FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION", "RUN_THE_HOOP_DRILL",
             "RUN_AND_CLUB_DRILL", "FOUR_BAG_AGILITY_DRILL", "PASS_RUSH_DRILL", "BODY_CONTROL_DRILL"]
f = per_drill(data.combine_tracking("SKILL_DRILLS_DL"), dl_ids, ["acel_c_max"]) \
    .filter(pl.col("drill_name").is_in(DL_DRILLS)).pivot(on="drill_name", index="nfl_id", values="acel_c_max")
f = f.with_columns(turn_force=zmean(f, DL_DRILLS, {}))
r = reg.join(dl_ids, on="nfl_id").filter(pl.col("player_get_off").is_not_null()).with_columns(
    edge=pl.col("lined_up_position").is_in(["EDGE", "OLB"]).cast(pl.Int8),
    pressed=pl.col("time_to_pressure").is_not_null().cast(pl.Int8))
r = r.with_columns(press_adj=pl.col("pressed") - pl.col("pressed").mean().over("lined_up_position"))
g = (r.group_by("nfl_id").agg(snaps=pl.len(), edge_share=pl.col("edge").mean(), s=pl.col("press_adj").sum())
     .with_columns(press_adj=pl.col("s") / (pl.col("snaps") + 100)).filter(pl.col("snaps") >= 50))
dl = (g.join(f.select("nfl_id", "turn_force"), on="nfl_id", how="left")
      .join(cr.select("nfl_id", "combine_weight", "ten_yd_split"), on="nfl_id").join(pl_, on="nfl_id")
      .with_columns(log_pick=pl.col("draft_overall_pick").fill_null(260).log()))
evaluate(dl, "press_adj", ["log_pick", "combine_weight", "edge_share"], "turn_force", "DL turn force")
evaluate(dl, "press_adj", ["log_pick", "combine_weight", "edge_share", "ten_yd_split"], "turn_force",
         "DL turn force (+10-yd split)")

# ---------------- WR: brake-and-burst ----------------
wr_ids = cr.filter(pl.col("combine_position") == "WR").select("nfl_id")
WR_BREAKS = ["DAGGER_ROUTE_LEFT", "CURL_ROUTE_RIGHT", "COMEBACK_ROUTE_RIGHT", "SLANT_ROUTE_LEFT",
             "SLOT_STRIKE_ROUTE_LEFT", "POST_CORNER_ROUTE_RIGHT", "SPEED_OUT_ROUTE_LEFT"]
fw = per_drill(data.combine_tracking("SKILL_DRILLS_WR"), wr_ids, ["decel", "reaccel", "min_s"]) \
    .filter(pl.col("drill_name").is_in(WR_BREAKS))
# per route: brake_burst = z(decel) + z(reaccel) - z(min_s), standardized within route
fw = fw.with_columns([((pl.col(c) - pl.col(c).mean().over("drill_name")) / pl.col(c).std().over("drill_name")).alias(c)
                      for c in ["decel", "reaccel", "min_s"]]) \
    .with_columns(bb=(pl.col("decel") + pl.col("reaccel") - pl.col("min_s")) / 3)
per_route = fw.pivot(on="drill_name", index="nfl_id", values="bb")
fw_all = fw.group_by("nfl_id").agg(brake_burst=pl.col("bb").mean(), n_routes=pl.len())
fw_nodag = fw.filter(pl.col("drill_name") != "DAGGER_ROUTE_LEFT").group_by("nfl_id").agg(brake_burst_nodagger=pl.col("bb").mean())
w = reg.join(wr_ids, on="nfl_id").filter(pl.col("separation_at_pass_forward").is_not_null())
w = w.with_columns(sep_adj=pl.col("separation_at_pass_forward") - pl.col("separation_at_pass_forward").mean().over("route_ran"))
gw = (w.group_by("nfl_id").agg(routes=pl.len(), s=pl.col("sep_adj").sum(),
                               slot_share=(pl.col("lined_up_position") == "SLOT_WR").mean())
      .with_columns(sep_adj=pl.col("s") / (pl.col("routes") + 50)).filter(pl.col("routes") >= 50))
wr = (gw.join(fw_all, on="nfl_id", how="left").join(fw_nodag, on="nfl_id", how="left")
      .join(cr.select("nfl_id", "combine_weight", "forty"), on="nfl_id").join(pl_, on="nfl_id")
      .with_columns(log_pick=pl.col("draft_overall_pick").fill_null(260).log()))
evaluate(wr, "sep_adj", ["log_pick", "combine_weight", "slot_share"], "brake_burst", "WR brake-and-burst (7 routes)")
evaluate(wr, "sep_adj", ["log_pick", "combine_weight", "slot_share"], "brake_burst_nodagger",
         "WR brake-and-burst WITHOUT the dagger route (robustness)")
evaluate(wr, "sep_adj", ["log_pick", "combine_weight", "slot_share", "forty"], "brake_burst",
         "WR brake-and-burst (+40-yd time)")
print("\nWR per-route correlation of brake_burst with sep_adj:")
pr = per_route.join(gw.select("nfl_id", "sep_adj"), on="nfl_id")
for c in WR_BREAKS:
    if c in pr.columns:
        s = pr.select(c, "sep_adj").drop_nulls()
        print(f"  {c:28s} n={s.height:3d} r={np.corrcoef(s[c], s['sep_adj'])[0, 1]:+.2f}")


In [ ]:
%%writefile sensitivity.py
"""Sensitivity of the CBS effect to arbitrary choices: shrinkage K, minimum rushes and the pick imputed for undrafted players."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
import numpy as np
import polars as pl
import statsmodels.formula.api as smf
import data

tab = pl.read_parquet(data.PROCESSED / "dl_cbs_table.parquet").filter(pl.col("cbs").is_not_null(), pl.col("snaps").is_not_null())
print(f"{'K':>4} {'min':>4} {'udfa':>5}  n   CBS +1SD (POE/100)  95% CI           p")
for K in (50, 100, 200):
    for mn in (30, 50, 100):
        for udfa in (224, 260, 300):
            d = (tab.filter(pl.col("snaps") >= mn)
                 .with_columns(poe=100 * (pl.col("pressures") - pl.col("exp_press")) / (pl.col("snaps") + K),
                               lp=pl.col("draft_overall_pick").fill_null(udfa).log())
                 .to_pandas())
            m = smf.wls("poe ~ cbs + lp + combine_weight + edge_share", d, weights=d["snaps"]).fit()
            lo, hi = m.conf_int().loc["cbs"]
            print(f"{K:>4} {mn:>4} {udfa:>5} {len(d):3d}   {10*m.params['cbs']:+.2f}              "
                  f"[{10*lo:+.2f}, {10*hi:+.2f}]   {m.pvalues['cbs']:.4f}")


In [ ]:
%%writefile figures.py
"""Writeup figures (PNG, light theme). Colorblind-safe palette (first three categorical slots, validated all-pairs)."""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import statsmodels.formula.api as smf
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
import data

FIG = data.ROOT / "figures"
FIG.mkdir(exist_ok=True)
PROC = data.PROCESSED

SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SEQ = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div", ["#c43c3c", "#e88d8d", "#f0efec", "#86b6ef", "#1c5cab"])
ARC = ["FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION", "RUN_THE_HOOP_DRILL", "RUN_AND_CLUB_DRILL"]

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"], "font.size": 11,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "lines.linewidth": 2, "axes.titlepad": 28, "savefig.dpi": 200, "savefig.bbox": "tight",
})


def subtitle(ax, text):
    ax.annotate(text, xy=(0, 1), xycoords="axes fraction", xytext=(0, 7), textcoords="offset points",
                color=INK2, fontsize=10, va="bottom")


tab = pl.read_parquet(PROC / "dl_cbs_table.parquet")
an = tab.filter(pl.col("snaps") >= 50, pl.col("cbs").is_not_null()).to_pandas()


# ---------- Fig 1: what bend looks like (Run the Hoop) ----------
def fig1():
    ct = data.combine_tracking("SKILL_DRILLS_DL").filter(pl.col("drill_name") == "RUN_THE_HOOP_DRILL")
    has = tab.filter(pl.col("RUN_THE_HOOP_DRILL").is_not_null(), pl.col("snaps") >= 200)
    hi = has.sort("RUN_THE_HOOP_DRILL", descending=True).row(0, named=True)
    lo = has.sort("RUN_THE_HOOP_DRILL").row(0, named=True)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.6), sharex=True, sharey=True, layout="constrained")
    vmax = 9
    for ax, p, lab in [(axes[0], hi, "More turn force"), (axes[1], lo, "Less turn force")]:
        g = ct.filter(pl.col("nfl_id") == p["nfl_id"])
        att = g.group_by("attempt").agg(pl.len()).sort("len", descending=True)["attempt"][0]
        g = g.filter(pl.col("attempt") == att)
        x, y, s = g["x"].to_numpy(), g["y"].to_numpy(), g["s"].to_numpy()
        x, y = x - x[0], y - y[0]
        h = np.unwrap(np.arctan2(np.gradient(y), np.gradient(x)))
        ac = np.convolve(s * np.abs(np.gradient(h)) / 0.1, np.ones(3) / 3, mode="same")
        ax.plot(x, y, color=GRID, lw=2, zorder=1)
        sc = ax.scatter(x, y, c=ac, cmap=SEQ, vmin=0, vmax=vmax, s=46, edgecolor=SURFACE, linewidth=1.2, zorder=2)
        i = int(np.argmax(ac))
        ax.annotate(f"peak {ac[i]:.1f} yd/s²", (x[i], y[i]), xytext=(12, -16), textcoords="offset points",
                    fontsize=10, color=INK, arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))
        ax.set_aspect("equal")
        ax.set_title(f"{lab}: {p['display_name']}", fontsize=12)
        subtitle(ax, f"CBS {p['cbs']:.0f} · {p['pressures']} pressures on {p['snaps']} NFL pass rushes")
        ax.set_xlabel("yards")
    axes[0].set_ylabel("yards")
    cb = fig.colorbar(sc, ax=axes, shrink=0.75, pad=0.02)
    cb.set_label("centripetal acceleration (yd/s²)", color=INK2)
    cb.outline.set_visible(False)
    fig.suptitle("Run the Hoop: the same arc to the eye, a different force to the sensor", x=0.02, ha="left", fontsize=15, fontweight="bold")
    fig.savefig(FIG / "fig1_hoop_paths.png")
    plt.close(fig)


# ---------- Fig 2: in-game mechanism ----------
def fig2():
    play = pl.read_parquet(PROC / "dl_rush_play_turnforce.parquet")
    reg = data.reg_season_plays().filter(pl.col("player_get_off").is_not_null()) \
        .select("game_id", "play_id", "nfl_id", "lined_up_position",
                pressed=pl.col("time_to_pressure").is_not_null().cast(pl.Int8))
    d = play.join(reg, on=["game_id", "play_id", "nfl_id"]).filter(
        pl.col("lined_up_position").is_in(["EDGE", "INTERIOR_LINE"]))
    edges = np.arange(0, 11, 1.0)
    fig, ax = plt.subplots(figsize=(9, 5))
    for pos, col, lab in [("EDGE", BLUE, "Edge"), ("INTERIOR_LINE", ORANGE, "Interior")]:
        g = d.filter(pl.col("lined_up_position") == pos).with_columns(
            b=pl.col("ac_max").clip(0, 10.99).floor()).group_by("b").agg(
            n=pl.len(), p=pl.col("pressed").mean()).filter(pl.col("n") >= 150).sort("b")
        xb, p, n = g["b"].to_numpy() + 0.5, g["p"].to_numpy(), g["n"].to_numpy()
        se = np.sqrt(p * (1 - p) / n)
        ax.fill_between(xb, 100 * (p - 1.96 * se), 100 * (p + 1.96 * se), color=col, alpha=0.15, lw=0)
        ax.plot(xb, 100 * p, color=col, marker="o", ms=8, mec=SURFACE, mew=2)
        ax.text(xb[-1] + 0.15, 100 * p[-1], lab, color=INK, va="center", fontsize=10)
    ax.set_xlim(0, 9.2)
    ax.set_xlabel("peak centripetal acceleration in the first 3 s after the snap (yd/s²)")
    ax.set_ylabel("pressures per 100 pass rushes")
    ax.set_title("On Sundays, rushers who turn harder reach the quarterback more")
    subtitle(ax, f"{d.height:,} rookie-contract DL pass rushes, 2023–25 regular season · band = 95% CI")
    ax.legend(handles=[mpl.lines.Line2D([], [], color=BLUE, marker="o", label="Edge"),
                       mpl.lines.Line2D([], [], color=ORANGE, marker="o", label="Interior")],
              frameon=False, loc="upper left")
    fig.savefig(FIG / "fig2_ingame_mechanism.png")
    plt.close(fig)


# ---------- Fig 3: main result by quartile ----------
def fig3():
    rng = np.random.default_rng(1)
    a = an.copy()
    a["q"] = np.digitize(a["cbs"], np.percentile(a["cbs"], [25, 50, 75])) + 1
    f = lambda d: 100 * (d["pressures"].sum() - d["exp_press"].sum()) / d["snaps"].sum()
    est = [f(a[a.q == q]) for q in range(1, 5)]
    boots = np.array([[f(s[s.q == q]) for q in range(1, 5)]
                      for s in (a.sample(len(a), replace=True, random_state=int(rng.integers(1e9))) for _ in range(2000))])
    lo, hi = np.percentile(boots, 2.5, axis=0), np.percentile(boots, 97.5, axis=0)
    fig, ax = plt.subplots(figsize=(8.5, 5))
    xs = np.arange(4)
    cols = [ORANGE if e < 0 else BLUE for e in est]
    ax.bar(xs, est, width=0.62, color=cols, edgecolor=SURFACE, linewidth=2)
    ax.errorbar(xs, est, yerr=[np.array(est) - lo, hi - np.array(est)], fmt="none", ecolor=INK2, elinewidth=1.2, capsize=4)
    for x, e in zip(xs, est):
        ax.text(x + 0.36, e, f"{e:+.1f}", ha="left", va="center", fontsize=11, color=INK, fontweight="bold")
    ax.axhline(0, color=AXIS, lw=1)
    ax.set_xticks(xs, ["Q1\nlowest CBS", "Q2", "Q3", "Q4\nhighest CBS"])
    ax.set_ylabel("pressures over expected\nper 100 pass rushes")
    ax.grid(axis="x", visible=False)
    ax.set_title("Higher Combine Bend Score, more NFL pressure")
    subtitle(ax, f"{len(a)} DL with ≥50 pass rushes · expected = alignment average (edge/interior) · bars = 95% bootstrap CI")
    fig.savefig(FIG / "fig3_quartiles.png")
    plt.close(fig)


# ---------- Fig 4: what it is worth (comparable coefficients) ----------
def fig4():
    d = an.dropna(subset=["ten_yd_split"]).copy()
    for c in ["cbs", "ten_yd_split", "log_pick", "combine_weight"]:
        d[c + "_z"] = (d[c] - d[c].mean()) / d[c].std()
    d["ten_fast_z"] = -d["ten_yd_split_z"]   # faster = positive
    d["pick_early_z"] = -d["log_pick_z"]     # earlier pick = positive
    m = smf.wls("poe100 ~ cbs_z + ten_fast_z + pick_early_z + combine_weight_z + edge_share", d, weights=d["snaps"]).fit()
    rows = [("Combine Bend Score", "cbs_z", BLUE), ("Draft slot (earlier)", "pick_early_z", MUTED),
            ("10-yd split (faster)", "ten_fast_z", MUTED), ("Weight", "combine_weight_z", MUTED)]
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    for i, (lab, k, col) in enumerate(rows[::-1]):
        b, (lo, hi) = m.params[k], m.conf_int().loc[k]
        ax.plot([lo, hi], [i, i], color=col, lw=2)
        ax.plot(b, i, "o", color=col, ms=10, mec=SURFACE, mew=2)
        ax.text(hi + 0.08, i, f"{b:+.2f}", va="center", fontsize=10, color=INK)
    ax.axvline(0, color=AXIS, lw=1)
    ax.set_yticks(range(len(rows)), [r[0] for r in rows[::-1]])
    ax.tick_params(axis="y", labelcolor=INK2, labelsize=11)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("pressures over expected per 100 pass rushes, per +1 SD")
    ax.set_title("Bend Score adds what the draft board and the stopwatch miss")
    subtitle(ax, f"Rush-weighted regression, n={int(m.nobs)} DL with a timed 10-yd split · lines = 95% CI")
    fig.savefig(FIG / "fig4_coefficients.png")
    plt.close(fig)


# ---------- Fig 5: out-of-sample validation ----------
def fig5():
    r = json.loads((PROC / "nested_dl.json").read_text())
    null = 100 * np.array(r["null"])
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    ax.hist(null, bins=40, color="#b7d3f6", edgecolor=SURFACE, linewidth=1)
    ax.axvline(100 * r["gain"], color=BLUE, lw=2.5)
    ax.text(100 * r["gain"], ax.get_ylim()[1] * 0.92, f"  Actual CBS: {100*r['gain']:+.1f}%\n  p = {r['p_perm']:.3f}",
            color=INK, va="top", fontsize=11, fontweight="bold")
    ax.set_xlabel("improvement predicting pressure for an unseen draft class (% error reduction)")
    ax.set_ylabel("permutations")
    ax.grid(axis="x", visible=False)
    ax.set_title("Honest validation: pick drills blind, test on a held-out draft class")
    subtitle(ax, f"Drill selection uses training classes only · light bars = {len(null)} random shuffles")
    fig.savefig(FIG / "fig5_validation.png")
    plt.close(fig)


# ---------- Fig 6: scouting board ----------
def fig6():
    d = tab.filter(pl.col("cbs").is_not_null()).to_pandas()
    d["x"] = d["draft_overall_pick"].fillna(290)
    played = d[d["snaps"].fillna(0) >= 50]
    rest = d[d["snaps"].fillna(0) < 50]
    fig, ax = plt.subplots(figsize=(10.5, 6))
    ax.scatter(rest["x"], rest["cbs"], s=40, facecolor="none", edgecolor=AXIS, linewidth=1.2, label="<50 NFL pass rushes")
    lim = np.nanmax(np.abs(played["poe100"]))
    sc = ax.scatter(played["x"], played["cbs"], c=played["poe100"], cmap=DIV, norm=TwoSlopeNorm(0, -lim, lim),
                    s=70, edgecolor=SURFACE, linewidth=1.5, zorder=3)
    ax.axhline(60, color=AXIS, lw=1)
    ax.axvline(100, color=AXIS, lw=1)
    ax.text(103, d["cbs"].max() + 1.2, "Hidden-talent zone: CBS ≥ 60 and pick > 100", color=INK2, fontsize=10)
    offsets = {"Cedric Johnson": (10, 18), "Javon Solomon": (-30, 26), "Adetomiwa Adebawore": (-14, -24),
               "Robert Beal": (25, -30), "Laiatu Latu": (14, 6), "Chop Robinson": (12, -16), "Tuli Tuipulotu": (14, 4)}
    for name, (dx, dy) in offsets.items():
        r = d[d["display_name"] == name]
        if len(r):
            r = r.iloc[0]
            label = "A. Adebawore" if name == "Adetomiwa Adebawore" else name
            ax.annotate(label, (r["x"], r["cbs"]), xytext=(dx, dy), textcoords="offset points", fontsize=9, color=INK,
                        ha="right" if dx < 0 else "left", va="center",
                        arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8, shrinkA=0, shrinkB=4))
    ax.set_xticks([1, 32, 64, 100, 150, 200, 250, 290], ["1", "32", "64", "100", "150", "200", "250", "UDFA"])
    ax.set_xlabel("overall draft pick")
    ax.set_ylabel("Combine Bend Score (50 = average)")
    cb = fig.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    cb.set_label("pressures over expected per 100 pass rushes", color=INK2)
    cb.outline.set_visible(False)
    ax.legend(frameon=False, loc="lower right")
    ax.set_title("Scouting board: Bend Score vs. draft slot")
    subtitle(ax, "Each dot is a 2023–25 DL prospect · blue = pressured above expectation, red = below")
    fig.savefig(FIG / "fig6_scouting_board.png")
    plt.close(fig)


# ---------- Fig 1 (animated): the same two Run the Hoop attempts in real time ----------
def fig1gif():
    from matplotlib.animation import FuncAnimation, PillowWriter
    ct = data.combine_tracking("SKILL_DRILLS_DL").filter(pl.col("drill_name") == "RUN_THE_HOOP_DRILL")
    has = tab.filter(pl.col("RUN_THE_HOOP_DRILL").is_not_null(), pl.col("snaps") >= 200)
    players = [has.sort("RUN_THE_HOOP_DRILL", descending=True).row(0, named=True),
               has.sort("RUN_THE_HOOP_DRILL").row(0, named=True)]
    tracks = []
    for p in players:
        g = ct.filter(pl.col("nfl_id") == p["nfl_id"])
        att = g.group_by("attempt").agg(pl.len()).sort("len", descending=True)["attempt"][0]
        g = g.filter(pl.col("attempt") == att)
        x, y, s = g["x"].to_numpy(), g["y"].to_numpy(), g["s"].to_numpy()
        x, y = x - x[0], y - y[0]
        h = np.unwrap(np.arctan2(np.gradient(y), np.gradient(x)))
        ac = np.convolve(s * np.abs(np.gradient(h)) / 0.1, np.ones(3) / 3, mode="same")
        tracks.append((x, y, ac))
    n = max(len(t[0]) for t in tracks)
    hold = 15  # 1.5 s pause on the final frame
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharex=True, sharey=True, layout="constrained")
    xs = np.concatenate([t[0] for t in tracks]); ys = np.concatenate([t[1] for t in tracks])
    arts = []
    for ax, p, (x, y, ac), lab in zip(axes, players, tracks, ["More turn force", "Less turn force"]):
        ax.set_xlim(xs.min() - 0.8, xs.max() + 0.8); ax.set_ylim(ys.min() - 0.8, ys.max() + 0.8)
        ax.set_aspect("equal")
        ax.plot(x, y, color=GRID, lw=2, zorder=1)
        ax.set_title(f"{lab}: {p['display_name']}", fontsize=12)
        subtitle(ax, f"CBS {p['cbs']:.0f} · {p['pressures']} pressures on {p['snaps']} NFL pass rushes")
        ax.set_xlabel("yards")
        sc = ax.scatter([], [], c=[], cmap=SEQ, vmin=0, vmax=9, s=46, edgecolor=SURFACE, linewidth=1.2, zorder=2)
        head, = ax.plot([], [], "o", ms=13, color=INK, mec=SURFACE, mew=2, zorder=3)
        txt = ax.text(0.97, 0.05, "", transform=ax.transAxes, fontsize=11, color=INK, fontweight="bold", ha="right")
        arts.append((sc, head, txt, x, y, ac))
    axes[0].set_ylabel("yards")
    cb = fig.colorbar(arts[0][0], ax=axes, shrink=0.75, pad=0.02)
    cb.set_label("centripetal acceleration (yd/s²)", color=INK2)
    cb.outline.set_visible(False)
    fig.suptitle("Run the Hoop in real time: same arc, different force", x=0.02, ha="left", fontsize=14, fontweight="bold")

    def update(i):
        out = []
        for sc, head, txt, x, y, ac in arts:
            k = min(i, len(x) - 1) + 1
            sc.set_offsets(np.c_[x[:k], y[:k]]); sc.set_array(ac[:k])
            head.set_data([x[k - 1]], [y[k - 1]])
            txt.set_text(f"now {ac[k - 1]:.1f}  ·  peak {ac[:k].max():.1f} yd/s²")
            out += [sc, head, txt]
        return out

    anim = FuncAnimation(fig, update, frames=n + hold, blit=False)
    anim.save(FIG / "fig1_hoop_paths.gif", writer=PillowWriter(fps=10), dpi=110)
    plt.close(fig)


if __name__ == "__main__":
    which = sys.argv[1:] or ["1", "1gif", "2", "3", "4", "5", "6"]
    for w in which:
        globals()[f"fig{w}"]()
        print("ok fig", w)


## 1. Robustness and in-game turn force

In [ ]:
%run validate_dl.py

## 2. Combine Bend Score and effect sizes

In [ ]:
%run final_metric.py

## 2b. Sensitivity to arbitrary choices

In [ ]:
%run sensitivity.py

## 3. Nested validation (takes ~15 min: 1,000 permutations)

In [ ]:
%run nested_test.py

## 4. Ideas that failed (reported for transparency)

In [ ]:
%run dl_viability.py
%run composite_test.py

## 5. Figures

In [ ]:
%run figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/*.png')):
    display(Image(f))